# 1. Introducing MCP

Model Context Protocol (MCP) is a communication layer that provides Claude with context and tools without requiring you to write a bunch of tedious integration code.

![](./figure/06/Introducing_MCP_01.jpg)

## Understanding MCP Through a Real Example

如果使用者透過 Github 問: "請問今天我的 repositories 全部加起來總共被要求幾次 pull?"，一種方法是自己寫 tool，包含 tool function 以及 schema

![](./figure/06/Introducing_MCP_02.jpg)

## The Tool Function Problem

但如果要自己寫，會花超多時間以及心力，因為 Github API 有超多功能，你得一個一個進行實作 tool functionality

![](./figure/06/Introducing_MCP_03.jpg)

## How MCP Solves This

MCP shifts the burden of tool definitions and execution from your server to MCP servers. Instead of you writing all those GitHub tools, they're **authored and executed inside a dedicated MCP server.**

![](./figure/06/Introducing_MCP_04.jpg)

The MCP server acts as a wrapper around GitHub's functionality, providing pre-built tools that you can use without having to implement them yourself.

MCP servers provide access to data or functionality implemented by outside services. They package up complex integrations into reusable components that any application can connect to.

![](./figure/06/Introducing_MCP_05.jpg)

## Common Questions About MCP

### Who Authors MCP Servers?
Anyone can create an MCP server implementation. Often, service providers themselves will make their own official MCP implementations. 

### How is MCP Different from Direct API Calls?
MCP servers provide tool schemas and functions already defined for you. If you call an API directly, you're responsible for authoring those tool definitions yourself. MCP saves you that implementation work.

### Isn't MCP Just Tool Use?
This is a common misconception. MCP servers and tool use are complementary but different concepts. MCP is about who does the work of creating and maintaining the tools. With MCP, someone else has already written the tool functions and schemas for you - they're packaged inside the MCP server.

![](./figure/06/Introducing_MCP_06.jpg)

# 2. MCP clients

## Transport Agnostic Communication

the client and server can talk to each other using different communication methods.

The most common setup runs both the MCP client and server on the same machine, where they communicate through standard input/output.

MCP clients and servers can also connect over:

- HTTP
- WebSockets
- Various other network protocols

![](./figure/06/MCP_Clients_01.jpg)

## Message Types

Once connected, the client and server exchange specific message types defined in the MCP specification. The main message types you'll work with are:
1. `ListToolsRequest / ListToolsResult`: The client asks the server "what tools do you provide?" and gets back a list of available tools.
2. `CallToolRequest / CallToolResult`: The client asks the server to run a specific tool with certain arguments, then receives the results.

![](./figure/06/MCP_Clients_02.jpg)
![](./figure/06/MCP_Clients_03.jpg)

## Complete Flow Example
A user asks *"What repositories do I have?"*

Here's the complete communication flow:

![](./figure/06/MCP_Clients_04.jpg)

# 3. Project

> 這部分直接去看 [這個資料夾](./MCP_cli_project_COMPLETE/)

這個專案中，我們要創建一個 MCP server that manages documents stored in memory. The server will provide two essential tools: 
1. one to read document contents
2. update them through find-and-replace operations.

![](./figure/06/Defining_Tools_with_MCP_01.jpg)

## Setting Up the MCP Server

我們先看 [mcp_server.py](./MCP_cli_project_COMPLETE/mcp_server.py) 這個程式碼的內容

可以使用以下程式碼初始化 MCP server

```python 
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("DocumentMCP", log_level="ERROR")
```

在這個 project 中，能夠被使用的檔案以簡單的字典形式儲存，`key` 是檔案名稱，`value` 是檔案說明

```python
docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project's budget and expenditure",
    "outlook.pdf": "This document presents the projected future performance of the",
    "plan.md": "The plan outlines the steps for the project's implementation.",
    "spec.txt": "These specifications define the technical requirements for the equipment"
}
```

## Tool Definition with Decorators

此 SDK 將繁瑣的工具建立流程簡化為簡潔易讀的流程。無需編寫冗長的 JSON 模式，而是使用 Python 裝飾器和類型提示。可以較簡單進行開發
> [裝飾器說明](https://steam.oxxostudio.tw/category/python/basic/decorator.html) 

## Creating the Document Reader Tool

首先我們先建立可以讓 Claude 可以依照 doc_id 來閱讀特定 file 的 tool，需要撰寫的內容如下:

---
```python
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string."
)
def read_document(
    doc_id: str = Field(description="Id of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    
    return docs[doc_id]
```

---

- The `@mcp.tool` decorator **automatically generates the JSON schema that Claude needs.**
- The Field class from `Pydantic` provides parameter descriptions that help Claude understand what each argument expects. (要先 ```from pydantic import Field```)

## Building the Document Editor Tool

The second tool performs simple find-and-replace operations on documents:

---

```python
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string."
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace."),
    new_str: str = Field(description="The new text to insert in place of the old text.")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)
```
---
This tool takes three parameters: 
1. the document ID, 
2. the text to find, and 
3. the replacement text. 

The implementation uses Python's built-in string `replace()` method for simplicity.

## Error Handling

Both tools include basic error handling to manage cases where Claude requests a document that doesn't exist. When an invalid document ID is provided, the tools raise a ValueError with a descriptive message that Claude can understand and potentially act upon.

# 4. The server inspector

When building MCP servers, you need a way to test your functionality without connecting to a full application.

The Python MCP SDK includes a built-in browser-based inspector that lets you debug and test your server in real-time.

## Starting the Inspector

This starts a development server on port `6277` and gives you a local URL to open in your browser. 

---
```bash
mcp dev mcp_server.py
```
---

## Connecting and Testing Tools

Click the "Connect" button on the left side to start your MCP server. Once connected, you'll see a navigation bar with sections for Resources, Prompts, Tools, and other features.

To test your tools:
- Navigate to the Tools section
- Click "List Tools" to see all available tools
- Select a tool to open its testing interface
- Fill in the required parameters
- Click "Run Tool" to execute and see results

## Development Workflow

The inspector creates an efficient development loop:

- Make changes to your MCP server code
- Test individual tools through the inspector
- Verify results without needing a full application setup
- Debug issues in isolation


This tool becomes essential as you build more complex MCP servers. It eliminates the need to wire up your server to Claude or another application just to test basic functionality, making development much faster and more focused.

# 5. Implementing a client

通常只會實作 Client 或 Server 其中之一，在這邊只是為了要比較好懂兩邊怎麼運作的才兩個都實作

## Understanding the Client Architecture

The MCP client consists of two main components:

- MCP Client : A custom class we create to make using the session easier
- Client Session : The actual connection to the server (part of the MCP Python SDK)

> The client session requires proper resource cleanup when we're done with it. That's why we wrap it in our custom MCP Client class - to handle all that cleanup automatically.

![](./figure/06/Implementing_a_Client_01.jpg)

## How the Client Fits Into Our Application

Our CLI code needs to do two main things with the MCP server:

1. Get a list of available tools to send to Claude
2. Execute tools when Claude requests them

The MCP client provides these capabilities through simple method calls that our application code can use.

![](./figure/06/Implementing_a_Client_02.jpg)

## Implementing the Core Methods

We need to implement two key methods in our client: `list_tools()` and `call_tool()`.

### List Tools Method

This method gets all available tools from the server:

```python
async def list_tools(self) -> list[types.Tool]:
    result = await self.session().list_tools()
    return result.tools
```

we access our session (the connection to the server), call the built-in `list_tools()` function, and return the `tools` from the result.

### Call Tool Method

This method executes a specific tool on the server:

```python
async def call_tool(
    self, tool_name: str, tool_input: dict
) -> types.CallToolResult | None:
    return await self.session().call_tool(tool_name, tool_input)
```

We pass the `tool name` and `input parameters` (provided by Claude) to the server and return the result.

## Testing the Client

To test our implementation, we can run the client directly. The file includes a testing harness that connects to our MCP server and calls our methods:

```python
async with MCPClient(
    command="uv", args=["run", "mcp_server.py"]
) as client:
    result = await client.list_tools()
    print(result)
```

When we run this test, we should see our tool definitions printed out, including the `read_doc_contents` and `edit_document tools` we created earlier.

```bash
uv run mcp_client.py #<- terminal 上運行
```

## Putting It All Together

Now that our client can list tools and call them, we can test the complete flow. When we run our main application and ask Claude about a document:

1. Our code uses the client to get available tools
2. These tools are sent to Claude along with the user's question
3. Claude decides to use the `read_doc_contents` tool
4. Our code uses the client to execute that tool
5. The result is sent back to Claude, who then responds to the user

For example, asking "What is the contents of the report.pdf document?" will trigger Claude to use our document reading tool, and we'll get back information about the 20m condenser tower document we set up in our server.

```bash
uv run main.py # <- 用這條指令在 terminal 上運行
```

The client acts as the bridge between our application logic and the MCP server, making it easy to access server functionality without worrying about the underlying connection details.

# 6. Defining resources

Resources in MCP servers allow you to **expose data to clients**, similar to GET request handlers in a typical HTTP server. They're perfect for scenarios where you need to fetch information rather than perform actions.

## Understanding Resources Through an Example

Let's say you want to build a document mention feature where users can type @document_name to reference files. This requires two operations:

- Getting a list of all available documents (for autocomplete)
- Fetching the contents of a specific document (when mentioned)

When a user types @, you need to show available documents. 
When they submit a message with a mention, you automatically inject that document's content into the prompt sent to Claude.

![](./figure/06/Defining_Resources_01.jpg)
![](./figure/06/Defining_Resources_02.jpg)

## How Resources Work

Resources follow a **request-response** pattern. Your client sends a `ReadResourceRequest` with a URI, and the MCP server responds with the data. The URI acts like an address for the resource you want to access.

![](./figure/06/Defining_Resources_03.jpg)

## Types of Resources

There are two types of resources:

- **Direct Resources**: Static URIs that don't change, like `docs://documents`
- **Templated Resources**: URIs with parameters, like `docs://documents/{doc_id}`. For templated resources, the Python SDK automatically parses parameters from the URI and passes them as keyword arguments to your function.

![](./figure/06/Defining_Resources_04.jpg)

## Implementing Resources

Resources are defined using the `@mcp.resource()` decorator. Here's how to create both types:

### 1. Direct Resource (List Documents)

```python
@mcp.resource(
    "docs://documents",
    mime_type="application/json"
)
def list_docs() -> list[str]:
    return list(docs.keys())
```

### 2. Templated Resource (Fetch Document)

```python
@mcp.resource(
    "docs://documents/{doc_id}",
    mime_type="text/plain"
)
def fetch_doc(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]
```

## MIME Types

Resources can return any type of data,
- strings, 
- JSON, 
- binary, etc. 

The mime_type parameter gives clients a hint about what kind of data you're returning:

- `application/json` - Structured JSON data
- `text/plain` - Plain text content
- Any other valid MIME type for different data formats

The MCP Python SDK automatically serializes your return values. You don't need to manually convert to JSON strings.

## Testing Resources

```bash
uv run mcp dev mcp_server.py
```

Then connect to the inspector in your browser. You'll see:

- Resources: Lists your direct/static resources
- Resource Templates: Shows templated resources that accept parameters

![](./figure/06/Defining_Resources_05.jpg)

Click on any resource to test it and see the exact response structure your client will receive.

![](./figure/06/Defining_Resources_06.jpg)

## Key Points

- `Resources` expose data, `Tools` perform actions
- Use direct resources for static data, templated resources for parameterized queries
- MIME types help clients understand response format
- The SDK handles serialization automatically
- Parameter names in templated URIs become function arguments

Resources provide a clean way to make data available to MCP clients, enabling features like document mentions, file browsing, or any scenario where you need to fetch information from your server.

# 7. Accessing resources
Resources in MCP allow your server to **expose data that can be directly included in prompts, rather than requiring tool calls to access information**. This creates a **more efficient** way to provide context to AI models like Claude.

## Understanding Resource Requests

When you've defined resources on your MCP server, your client needs a way to request and use them. The client acts as a bridge between your application and the MCP server, handling the communication and data parsing automatically.

The flow is straightforward: when a user wants to reference a document (like typing "`@report.pdf`"), your application uses the MCP client to fetch that resource from the server and include its contents directly in the prompt sent to Claude.

![](./figure/06/Accessing_Resources_01.jpg)

## Implementing Resource Reading

The core functionality requires a `read_resource` function in your **MCP client**. This function takes a `URI` parameter identifying which resource to fetch:
```python
async def read_resource(self, uri: str) -> Any:
    result = await self.session().read_resource(AnyUrl(uri))
    resource = result.contents[0]
```
The response from the MCP server contains a contents list. You typically only need the first element, which contains the actual resource data along with metadata like the MIME type.

## Required Imports

```python
import json
from pydantic import AnyUrl
```

## Handling Different Content Types

Resources can return different types of content, so your client needs to parse them appropriately. The **`MIME`** type tells you how to handle the data:
```python
if isinstance(resource, types.TextResourceContents):
    if resource.mimeType == "application/json":
        return json.loads(resource.text)
    
    return resource.text
```
This approach ensures that JSON resources are properly parsed into Python objects, while plain text resources are returned as strings. 

The `MIME` type acts as your hint for determining the correct parsing strategy.

## Testing Resource Access

```bash
uv run mcp_client.py
```
Once implemented, you can test the functionality through your CLI application. When you type something like "What's in the @report.pdf document?", the system should:

- Show available resources in an autocomplete list
- Allow you to select a resource
- Fetch the resource content automatically
- Include that content in the prompt to Claude

The key advantage is that Claude receives the document content **directly in the prompt, eliminating the need for tool calls to access the information**. This makes interactions faster and more efficient.

![](./figure/06/Accessing_Resources_02.jpg)

# 8.Defining prompts

Prompts in **MCP servers** let you define pre-built, high-quality instructions that clients can use instead of writing their own prompts from scratch. Think of them as carefully crafted templates that give better results than what users might come up with on their own.

## Why Use Prompts?

Let's say you want Claude to reformat a document into markdown. A user could just type "convert report.pdf to markdown" and it would work fine. But they'd probably get much better results with a thoroughly tested prompt that includes specific instructions about formatting, structure, and output requirements.

The key insight is that while users can accomplish these tasks on their own, they'll get more consistent and higher-quality results when using prompts that have been carefully developed and tested by the MCP server authors.

![](./figure/06/Defining_Prompts_01.jpg)

## How Prompts Work

Prompts define **a set of user and assistant messages that clients can use directly**. When a client requests a prompt, your server returns a *list* of messages that can be sent straight to Claude.

The basic structure looks like this:
- Define prompts using the `@mcp.prompt()` decorator
- Add a name and description for each prompt
- Return a list of messages that form the complete prompt
- These prompts should be high quality, well-tested, and relevant to your MCP server's purpose

![](./figure/06/Defining_Prompts_02.jpg)

## Building a Format Command

Here's how to implement a document formatting prompt. First, you'll need to import the base message types:
```python
from mcp.server.fastmcp import base
```

Then define your prompt function:
```python
@mcp.prompt(
    name="format",
    description="Rewrites the contents of the document in Markdown format."
)
def format_document(
    doc_id: str = Field(description="Id of the document to format")
) -> list[base.Message]:
    prompt = f"""
Your goal is to reformat a document to be written with markdown syntax.

The id of the document you need to reformat is:

{doc_id}


Add in headers, bullet points, tables, etc as necessary. Feel free to add in extra formatting.
Use the 'edit_document' tool to edit the document. After the document has been reformatted...
"""
    
    return [
        base.UserMessage(prompt)
    ]
```

## Testing Your Prompts

You can test prompts using the MCP Inspector. Navigate to the Prompts section, select your prompt, and provide any required parameters. The inspector will show you the generated messages that would be sent to Claude.

This lets you verify that your prompt interpolates variables correctly and produces the expected message structure before using it in a real application.

![](./figure/06/Defining_Prompts_03.jpg)

## Best Practices

When creating prompts for your MCP server:

- Focus on tasks that are central to your server's purpose
- Write detailed, specific instructions rather than vague requests
- Test your prompts thoroughly with different inputs
- Include clear descriptions so users understand what each prompt does
- Consider how the prompt will work with your server's tools and resources

Remember that prompts are meant to provide value that users couldn't easily get on their own - they should represent your expertise in the domain your MCP server covers.

## Integration with Your Application

Remember that the MCP client code you write gets used by other parts of your application. The `read_resource` function becomes a building block that other components can call to fetch document contents, list available resources, or integrate resource data into prompts.

This separation of concerns keeps your code clean: the MCP client handles communication with the server, while your application logic focuses on how to use that data effectively.

# 9. Prompts in the client

Prompts in MCP define a set of user and assistant messages that can be used by the client. These prompts should be high quality, well-tested, and relevant to the overall purpose of the MCP server.

![](./figure/06/Prompts_in_the_Client_01.jpg)

## Implementing List Prompts

The first step is implementing the `list_prompts` method in your MCP client. This method **retrieves all available prompts from the server**:
```python
async def list_prompts(self) -> list[types.Prompt]:
    result = await self.session().list_prompts()
    return result.prompts
```
This simple implementation calls the session's `list_prompts` method and returns the prompts array from the result.

## Getting Individual Prompts

The `get_prompt` method retrieves a specific prompt with arguments interpolated into it. When you request a prompt, you provide arguments that get passed to the prompt function as keyword arguments:
```python
async def get_prompt(self, prompt_name, args: dict[str, str]):
    result = await self.session().get_prompt(prompt_name, args)
    return result.messages
```
The method returns the messages from the result, which **form a conversation that can be fed directly into Claude**.

## How Prompt Arguments Work

When you define a prompt function on the server side, it can accept parameters. For example, a document formatting prompt might expect a `doc_id` parameter:
```python
def format_document(doc_id: str):
    # The doc_id gets interpolated into the prompt
```
When the client calls `get_prompt`, the arguments dictionary **should contain the expected keys**. The MCP server will pass these as keyword arguments to the prompt function, allowing **dynamic content to be inserted into the prompt template**.

## Testing Prompts in the CLI

Once implemented, you can test prompts through the command-line interface. When you type a `forward slash` (`/`), available prompts appear as commands. Selecting a prompt may prompt you to choose from available options (like document IDs), and then the complete prompt gets sent to Claude.

The workflow looks like this:

1. User selects a prompt (like "format")
2. System prompts for required arguments (like which document to format)
3. The prompt gets sent to Claude with the interpolated values
4. Claude can then use tools to fetch additional data and complete the task

![](./figure/06/Prompts_in_the_Client_02.jpg)
![](./figure/06/Prompts_in_the_Client_03.jpg)

## Prompt Best Practices

When creating prompts for your MCP server:

- Make them relevant to your server's purpose
- Test them thoroughly before deployment
- Use clear, specific instructions
- Design them to work well with your available tools
- Consider what arguments users will need to provide

Prompts bridge the gap between predefined functionality and dynamic user needs, giving Claude structured starting points for complex tasks while maintaining flexibility through parameterization.

# 10. MCP Review

![](./figure/06/MCP_review.jpg)

# Extra : 如何做到 client 與 server 通信 (這部分內容是與 Claude 討論產出)

![](./figure/06/mcp_resource_flow.svg)

### 背後的核心：兩個 JSON-RPC 呼叫
當你在 client 輸入 `@` 時，實際上觸發的是 MCP 協議 的兩個標準方法，而不是你直接跟 server 的函式溝通。
1. `resources/list` — 問「你有哪些資源？」
    Client 輸入 `@` 後，會立即發一個 JSON-RPC 請求給 server：
    ```json
    json{ "method": "resources/list", "params": {} }
    ```
    Server 回傳所有已註冊的資源 URI 列表，client 就用這些資料渲染出你看到的下拉選單。

2. `resources/read` — 問「給我這個資源的內容」
    你從選單選了某個 URI 後，client 再發第二個請求：
    ```json
    json{ "method": "resources/read", "params": { "uri": "file:///config.txt" } }
    ```
    Server 把對應的內容（文字或 `base64 binary`）回傳，client 把它塞進對話的 context 裡，LLM 就能讀到了。

### `@mcp.resource` 裝飾器的角色
你的問題問得很好！裝飾器本身不是在請求時才執行的，它的工作**在 server 啟動時**就完成了：
```python
@mcp.resource("file:///config.txt")
def get_config() -> str:
    return open("config.txt").read()
```
這段程式碼在 server 啟動的瞬間，`@mcp.resource` 就把這個 URI 和 `get_config` 函式的對應關係存進 server 的內部 registry（一個 dict）。
之後：

- 收到 resources/list → 把 registry 裡所有 URI 回傳
- 收到 resources/read → 查 registry 找到對應函式 → 呼叫它 → 回傳結果

所以 @mcp.resource 本質上是一個路由註冊器，跟 Flask/FastAPI 的 @app.route 概念一模一樣。

### 傳輸層：怎麼「連」到 server？
這些 JSON-RPC 訊息走的是兩種傳輸方式之一：

- stdio：本地 server，client 直接 spawn 一個子 process，透過 stdin/stdout 通訊
- SSE（Server-Sent Events）：遠端 server，走 HTTP

Client 和 server 在建立連線時會先做 handshake（initialize 方法），協商版本和能力，之後才開始正式的 resources/list 等呼叫。

### 一句話總結
>輸入 @ → client 發 resources/list → server 查裝飾器建立的 registry → 回傳 URI 列表 → 你選一個 → client 發 resources/read → server 執行對應函式 → 回傳內容 → 注入 LLM context

@mcp.resource 只是負責「啟動時登記」，真正的資料交換靠的是 MCP 協議的兩個標準方法。這也是為什麼不同語言的 MCP SDK 都能互通——因為協議層是統一的。

# Extra

這就是問題，client 是怎麼知道要 call  resources/list 來取得參數? 難不成 server.py 裡面有寫嗎?請你思考後明確說明給我是哪裡得來的資訊或是哪個部份的程式碼讓 client 可以知道要做 resources/list
```python
from mcp.server.fastmcp import FastMCP
from pydantic import Field
from mcp.server.fastmcp.prompts import base

mcp = FastMCP("DocumentMCP", log_level="ERROR")

docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project's budget and expenditures.",
    "outlook.pdf": "This document presents the projected future performance of the system.",
    "plan.md": "The plan outlines the steps for the project's implementation.",
    "spec.txt": "These specifications define the technical requirements for the equipment.",
}

# TODO: Write a tool to read a doc
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string.",
)
def read_document(
    doc_id: str = Field(description="Id of the document to read"),
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    return docs[doc_id]

# TODO: Write a tool to edit a doc
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string",
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(
        description="The text to replace. Must match exactly, including whitespace"
    ),
    new_str: str = Field(
        description="The new text to insert in place of the old text"
    ),
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    docs[doc_id] = docs[doc_id].replace(old_str, new_str)

# TODO: Write a resource to return all doc id's
@mcp.resource("docs://documents", mime_type="application/json")
def list_docs() -> list[str]:
    return list(docs.keys())

# TODO: Write a resource to return the contents of a particular doc
@mcp.resource("docs://documents/{doc_id}", mime_type="text/plain")
def fetch_doc(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]

# TODO: Write a prompt to rewrite a doc in markdown format
@mcp.prompt(
    name="format",
    description="Rewrites the contents of the document in Markdown format.",
)
def format_document(
    doc_id: str = Field(description="Id of the document to format"),
) -> list[base.Message]:
    prompt = f"""
    Your goal is to reformat a document to be written with markdown syntax.

    The id of the document you need to reformat is:
    <document_id>
    {doc_id}
    </document_id>

    Add in headers, bullet points, tables, etc as necessary. Feel free to add in extra text, but don't change the meaning of the report.
    Use the 'edit_document' tool to edit the document. After the document has been edited, respond with the final version of the doc. Don't explain your changes.
    """

    return [base.UserMessage(prompt)]

# TODO: Write a prompt to summarize a doc

if __name__ == "__main__":
    mcp.run(transport="stdio")
```

課程中也有這一段話，但我不認為他有明確表達什麼內容，但我在這裡提供你參考

<course_content>

## How Prompt Arguments Work

When you define a prompt function on the server side, it can accept parameters. For example, a document formatting prompt might expect a doc_id parameter:

def format_document(doc_id: str):
    # The doc_id gets interpolated into the prompt

When the client calls get_prompt, the arguments dictionary should contain the expected keys. The MCP server will pass these as keyword arguments to the prompt function, allowing dynamic content to be inserted into the prompt template.

## Testing Prompts in the CLI

Once implemented, you can test prompts through the command-line interface. When you type a `forward slash` (`/`), available prompts appear as commands. Selecting a prompt may prompt you to choose from available options (like document IDs), and then the complete prompt gets sent to Claude.

The workflow looks like this:

1. User selects a prompt (like "format")
2. System prompts for required arguments (like which document to format)
3. The prompt gets sent to Claude with the interpolated values
4. Claude can then use tools to fetch additional data and complete the task
</course_content>

<course_content_2>
Remember that the MCP client code you write gets used by other parts of your application. The `read_resource` function becomes a building block that other components can call to fetch document contents, list available resources, or integrate resource data into prompts.

This separation of concerns keeps your code clean: the MCP client handles communication with the server, while your application logic focuses on how to use that data effectively.
</course_content_2>

content_2 的部分我需要你閱讀，說明 application logic 會設計在哪裡?不是都放在 MCP client 裡嗎?